In [1]:
import torch
checkpoint = torch.load('./model.ckpt.pt')
checkpoint


{'model': OrderedDict([('_extra_state',
               {'model_params': {'type': 'matris',
                 'type_map': ['O', 'H'],
                 'pairwise_cutoff': 6.0,
                 'three_body_cutoff': 4.5,
                 'sel': 120,
                 'num_layers': 6,
                 'node_feat_dim': 64,
                 'edge_feat_dim': 64,
                 'three_body_feat_dim': 64,
                 'num_radial': 5,
                 'num_angular': 5,
                 'is_intensive': True,
                 'is_conservation': True,
                 'data_stat_nbatch': 10,
                 'data_stat_protect': 0.01,
                 'data_bias_nsample': 10,
                 'pair_exclude_types': [],
                 'atom_exclude_types': [],
                 'preset_out_bias': None,
                 'srtab_add_bias': True,
                 'mlp_hidden_dims': [128, 256, 128],
                 'dropout': 0.0,
                 'use_bias': False,
                 'distance_expans

In [ ]:
from matris.model import MatRIS
from matris.graph import GraphConverter


from matris.graph import GraphConverter, RadiusGraph
from pymatgen.core import Structure, Lattice

def dp_tensor_to_matris_graph(
    coord: torch.Tensor,      # (n_atoms, 3) 
    atype: torch.Tensor,      # (n_atoms,)
    cell: torch.Tensor,       # (3, 3) 
    cutoff: float = 6.0,
    three_body_cutoff: float = 4.0,
    graph_id: str = None,
):
    """

    """
   
    positions = coord.detach().cpu().numpy()
    atom_types = atype.detach().cpu().numpy()
    
    if cell.numel() == 9:
        lattice_vec = cell.view(3, 3).detach().cpu().numpy()
    else:
        lattice_vec = cell.detach().cpu().numpy()
    
    # 
    species = [type_map[int(t)] for t in atom_types]
    lattice = Lattice(lattice_vec)
    structure = Structure(
        lattice=lattice,
        species=species,
        coords=positions,
        coords_are_cartesian=True
    )
    
    # 3. Structure → RadiusGraph
    graph_converter = GraphConverter(
        atom_graph_cutoff=cutoff,
        line_graph_cutoff=three_body_cutoff,
    )
    
    graph = graph_converter(structure, graph_id=graph_id)
    return graph


def dp_batch_to_matris_graphs(
    coord: torch.Tensor,      # (batch, n_atoms, 3)
    atype: torch.Tensor,      # (batch, n_atoms)
    cell: torch.Tensor,       # (batch, 9) 
    type_map: list[str],
    cutoff: float = 6.0,
    three_body_cutoff: float = 4.0,
):

    batch_size = coord.shape[0]
    graphs = []
    
    for i in range(batch_size):
        graph = dp_tensor_to_matris_graph(
            coord=coord[i],
            atype=atype[i],
            cell=cell[i] if cell is not None else None,
            type_map=type_map,
            cutoff=cutoff,
            three_body_cutoff=three_body_cutoff,
            graph_id=f"batch_{i}",
        )
        graphs.append(graph)
    
    return graphs




import dpdata
import torch


data = dpdata.LabeledSystem("./test_data_h2o/", fmt="deepmd/npy")


coord = torch.tensor(data["coords"], dtype=torch.float32)
atype = torch.tensor([data['atom_types']] * len(data), dtype=torch.long)  
box = torch.tensor(data['cells'].reshape(-1, 9), dtype=torch.float32)
type_map = data.get_atom_names()
graphs_deepmd = dp_batch_to_matris_graphs(coord, atype, box, type_map) # 


/tmp/ipykernel_3838572/1331154050.py:95: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:254.)
  atype = torch.tensor([data['atom_types']] * len(data), dtype=torch.long)


In [ ]:
from matris.model import MatRIS
from matris.graph import GraphConverter


from matris.graph import GraphConverter, RadiusGraph
from pymatgen.core import Structure, Lattice
import numpy as np
def dp_tensor_to_matris_graph(
    coord: torch.Tensor,      # (n_atoms, 3) 
    atype: torch.Tensor,      # (n_atoms,)
    cell: torch.Tensor,       # (3, 3)
    type_map: list[str],
):
    """
    """
    #
    batch_size, n_atoms = atype.shape
    atoms_list = []
    
    for batch_idx in range(batch_size):
        frame_coord = coord[batch_idx].detach().cpu().numpy() # gradient break
        frame_atype = atype[batch_idx].detach().cpu().numpy() # gradient break
        symbols = [type_map[int(t)] for t in frame_atype]
    
        # 2. 
        species = [type_map[int(t)] for t in frame_atype]
        if cell is not None:
            if cell.dim() == 2 and cell.shape[1] == 9:
                # (batch_size, 9) -> (3, 3)
                frame_cell = cell[batch_idx].view(3, 3).detach().cpu().numpy()
            else:
                # (batch_size, 3, 3)
                frame_cell = cell[batch_idx].detach().cpu().numpy()
        lattice = Lattice(frame_cell)
        
        structure = Structure(
            lattice=lattice,
            species=species,
            coords=frame_coord,
            coords_are_cartesian=True
        )
        atoms_list.append(structure)
    return atoms_list
def dp_batch_to_matris_graphs(
    coord: torch.Tensor,      # (batch, n_atoms, 3)
    atype: torch.Tensor,      # (batch, n_atoms)
    cell: torch.Tensor,       # (batch, 9) 
    type_map: list[str],
    cutoff: float = 6.0,
    three_body_cutoff: float = 4.0,
):
    batch_size = coord.shape[0]
    graphs = []
    atom_list = dp_tensor_to_matris_graph(coord, atype, cell, type_map)
    for atom_i in atom_list:
        graph = GraphConverter(
            atom_graph_cutoff=cutoff,
            line_graph_cutoff=three_body_cutoff,
        )(atom_i)
        graphs.append(graph)
    
    return graphs   




import dpdata
import torch

# 使用 dpdata.DeepMD 读取 DeepMD-kit 格式的数据
data = dpdata.LabeledSystem("./test_data_h2o/", fmt="deepmd/npy")

#print(data)
## 和deepmd-kit保持一致，把数据变成tensor， 模拟forward函数的输入
coord = torch.tensor(data["coords"], dtype=torch.float32)
atype = torch.tensor([data['atom_types']] * len(data), dtype=torch.long)  
box = torch.tensor(data['cells'].reshape(-1, 9), dtype=torch.float32)
type_map = data.get_atom_names()
graphs_deepmd = dp_batch_to_matris_graphs(coord, atype, box, type_map) # 33.8s这个速度是差不多的，只差了3 s不能算慢很多了。


In [ ]:
import torch
import numpy as np
import dpdata
import sys

# 添加 MatRIS 路径
sys.path.insert(0, '/aisi/mnt/data_nas/jwzhou/opt/MatRIS')

from matris.model import MatRIS
from matris.graph import GraphConverter
from pymatgen.core import Structure, Lattice
import time

model_params = {
    "pairwise_cutoff": 6.0,
    "three_body_cutoff": 4.5,
    "num_layers": 6,
    "node_feat_dim": 64,
    "edge_feat_dim": 64,
    "three_body_feat_dim": 64,
    "mlp_hidden_dims": [128, 256, 128],  # 
    "dropout": 0.0,
    "use_bias": False,
    "distance_expansion": "Bessel",
    "three_body_expansion": "fourier",  # 
    "num_radial": 5,
    "num_angular": 5,
    "max_l": 4,
    "max_n": 4,
    "envelope_exponent": 8,
    "graph_conv_mlp": "gatemlp",
    "activation_type": "silu",
    "norm_type": "layer",   # 
    "use_smoothed_for_delta_edge": True,
    "learnable_basis": True,
    "is_intensive": False,
    "is_conservation": True,
    "reference_energy": None,  # 不使用参考能量
}
matris_model = MatRIS(**model_params)
device = "cuda"
matris_model = matris_model.to(device)
matris_model = matris_model.to(torch.float32)
checkpoint = torch.load('./model.ckpt.pt', map_location=device)
full_state = checkpoint['model']
matris_state = {
    k.replace("model.Default.matris_model.", "") : v for k, v in full_state.items() if k.startswith("model.Default.matris_model.")
} # 重新定义matris_state字典，去掉前缀
matris_model.load_state_dict(matris_state)
graph = graphs_deepmd[0].to(device)

t2 = time.time()
result = matris_model([graph], task="ef", is_training=False)
print(result)



MatRIS initialized with 1655182 parameters


/tmp/ipykernel_3838572/2362513684.py:44: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  checkpoint = torch.load('./model.ckpt.pt', map_location=device)


{'f': (tensor([[-0.0008, -0.0000, -0.0058],
        [ 0.0075, -0.0000,  0.0032],
        [-0.0067, -0.0000,  0.0026]], device='cuda:0'),), 'e': tensor([-0.1586], device='cuda:0', grad_fn=<AsStridedBackward0>), 'atoms_per_graph': tensor([3], device='cuda:0', dtype=torch.int32), 'ref_energy': 0}


In [5]:
matris_model.load_state_dict(checkpoint['model_state_dict'])


KeyError: 'model_state_dict'